In [2]:
import random
import torch
import matplotlib.pyplot as plt

In [3]:
## 数据生成
def data_generation(w, b, sample_size):
    X = torch.normal(0, 1, (sample_size, len(w))) 

    y = torch.matmul(X, w) + b

    y += torch.normal(0, 0.01, y.shape)

    return X, y.reshape(-1, 1)


true_w = torch.tensor([2.0, -3.4])
true_b = 4.2

features, labels = data_generation(true_w, true_b, 1000)


In [4]:
## 数据抽取-- random small batch for learning
def data_iter(batch_size,features,labels):
    sample_size = len(labels) ## 获取sample_size, 用len()获取数据的第一维度
    ## 生成index list
    indices = list(range(sample_size))
    ## shuffle indices
    random.shuffle(indices)
    for i in range(0,sample_size, batch_size): ## step_size就是batch_size
        batch_indices = torch.tensor(indices[i:min(i+batch_size,sample_size)])
    yield features[batch_indices],labels[batch_indices] ## 使用yield反复获取batch

In [5]:
## parameter initialization
w = torch.normal(0,0.01,size=(2,1), requires_grad = True)
b = torch.zeros(1, requires_grad = True)

In [6]:
## define linear regression model
def linear_regression(X,w,b):
    return torch.matmul(X,w) + b

In [7]:
## define loss function
def MSE(y_hat,y):
    return 0.5 *(y_hat - y.reshape(y_hat.shape))**2

In [8]:
## sgd
def SGD(lr,batch_size,params):
    with torch.no_grad():
        for param in params:
            param -= (lr/batch_size)*param.grad
            param.grad.zero_()

In [9]:
## setting hyperparameters
nums_epoch = 3
lr = 0.03
batch_size = 100

In [106]:
## training model

for epoch in range(nums_epoch):

    for X, y in data_iter(batch_size, features, labels):

        loss = MSE(
            linear_regression(X, w, b),
            y
        )

        ## 计算梯度
        loss.sum().backward()

        ## 更新参数
        SGD(lr, batch_size, [w, b])

    ## 每个 epoch 结束评估一次
    with torch.no_grad():

        train_loss = MSE(
            linear_regression(features, w, b),
            labels
        )

        print(
            f"epoch {epoch + 1}, "
            f"loss {train_loss.mean().item():f}"
        )

epoch 1, loss 0.000049
epoch 2, loss 0.000049
epoch 3, loss 0.000049
